# Importación de librerías y carga de datos base

In [1]:

import pandas as pd

df_info = pd.read_csv('./../../dataset/oulad/studentInfo.csv')
df_vle = pd.read_csv('./../../dataset/oulad/studentVle.csv')

print("Datos cargados correctamente para CP-07.")

Datos cargados correctamente para CP-07.


# Definición de la variable objetivo y partición cronológica

In [2]:
df_info['target'] = df_info['final_result'].apply(lambda x: 1 if x == 'Withdrawn' else 0)

presentaciones_2013 = ['2013B', '2013J']
presentaciones_2014 = ['2014B', '2014J']

df_train_info = df_info[df_info['code_presentation'].isin(presentaciones_2013)].copy()
df_test_info = df_info[df_info['code_presentation'].isin(presentaciones_2014)].copy()

print(f"Estudiantes históricos (Train 2013): {len(df_train_info)}")
print(f"Estudiantes nuevos (Test 2014): {len(df_test_info)}")

Estudiantes históricos (Train 2013): 13529
Estudiantes nuevos (Test 2014): 19064


# Función de extracción de características temporales

In [3]:
def preparar_datos_temporales(df_vle, df_info_split, max_dias):
    vle_filtrado = df_vle[df_vle['date'] <= max_dias]

    vle_agg = vle_filtrado.groupby(['id_student', 'code_module', 'code_presentation'])['sum_click'].sum().reset_index()
    vle_agg.rename(columns={'sum_click': 'total_clicks'}, inplace=True)

    df_merged = pd.merge(df_info_split, vle_agg, on=['id_student', 'code_module', 'code_presentation'], how='left')
    df_merged['total_clicks'] = df_merged['total_clicks'].fillna(0)

    X = df_merged[['total_clicks']]
    y = df_merged['target']

    return X, y

print("Función de preparación de datos lista.")

Función de preparación de datos lista.


# Ejecución del experimento completo (Random Forest)

In [4]:
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import ADASYN

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self

    def predict(self, X):
        proba = self.estimator.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

scoring_metrics = {
    'Accuracy': make_scorer(accuracy_score),
    'Precision': make_scorer(precision_score, zero_division=0),
    'Recall': make_scorer(recall_score, zero_division=0),
    'F1-Score': make_scorer(f1_score, zero_division=0)
}

param_grid_rf = {
    'clasificador__estimator__max_depth': [3, 5, None],
    'clasificador__estimator__min_samples_leaf': [1, 5],
    'clasificador__threshold': [0.3, 0.4, 0.5, 0.6]
}

ventanas_temporales = [30, 60, 90]
estrategias_balanceo = ['Línea Base (Sin balanceo)', 'Undersampling', 'ADASYN']

resultados_mejores = []
todas_las_permutaciones = []

print("Iniciando búsqueda exhaustiva CP-07 (Random Forest)...")

for dias in ventanas_temporales:
    print(f"--- Evaluando ventana a {dias} días ---")

    X_train, y_train = preparar_datos_temporales(df_vle, df_train_info, dias)
    X_test, y_test = preparar_datos_temporales(df_vle, df_test_info, dias)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for estrategia in estrategias_balanceo:
        if estrategia == 'Undersampling':
            sampler = RandomUnderSampler(random_state=42)
        elif estrategia == 'ADASYN':
            sampler = ADASYN(random_state=42, n_neighbors=5)
        else:
            sampler = None

        pasos = []
        if sampler is not None:
            pasos.append(('sampler', sampler))

        rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        wrapper = ThresholdWrapper(estimator=rf_base)
        pasos.append(('clasificador', wrapper))

        pipeline = ImbPipeline(pasos)

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid_rf,
            cv=5,
            scoring=scoring_metrics,
            refit='Recall',
            n_jobs=-1,
            return_train_score=False
        )

        grid_search.fit(X_train_scaled, y_train)

        mejor_modelo = grid_search.best_estimator_
        y_pred = mejor_modelo.predict(X_test_scaled)

        resultados_mejores.append({
            'Días': dias,
            'Estrategia': estrategia,
            'Mejor Max_Depth': str(grid_search.best_params_['clasificador__estimator__max_depth']),
            'Mejor Min_Samples_Leaf': grid_search.best_params_['clasificador__estimator__min_samples_leaf'],
            'Mejor Threshold': grid_search.best_params_['clasificador__threshold'],
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'F1-Score': f1_score(y_test, y_pred, zero_division=0)
        })

        cv_results = grid_search.cv_results_
        for i in range(len(cv_results['params'])):
            todas_las_permutaciones.append({
                'Días': dias,
                'Estrategia': estrategia,
                'Max_Depth': str(cv_results['param_clasificador__estimator__max_depth'][i]),
                'Min_Samples_Leaf': cv_results['param_clasificador__estimator__min_samples_leaf'][i],
                'Threshold': cv_results['param_clasificador__threshold'][i],
                'CV_Accuracy': cv_results['mean_test_Accuracy'][i],
                'CV_Precision': cv_results['mean_test_Precision'][i],
                'CV_Recall': cv_results['mean_test_Recall'][i],
                'CV_F1': cv_results['mean_test_F1-Score'][i]
            })

        print(f"  > {estrategia}: Procesado.")

df_mejores = pd.DataFrame(resultados_mejores)
df_todas_permutaciones = pd.DataFrame(todas_las_permutaciones)

mejor_fila = df_mejores.sort_values(by=['Recall', 'F1-Score'], ascending=[False, False]).iloc[0]
mejor_dia = mejor_fila['Días']
mejor_estrategia = mejor_fila['Estrategia']
mejor_depth = mejor_fila['Mejor Max_Depth']
mejor_leaf = mejor_fila['Mejor Min_Samples_Leaf']

df_tradeoff = df_todas_permutaciones[
    (df_todas_permutaciones['Días'] == mejor_dia) &
    (df_todas_permutaciones['Estrategia'] == mejor_estrategia) &
    (df_todas_permutaciones['Max_Depth'] == mejor_depth) &
    (df_todas_permutaciones['Min_Samples_Leaf'] == mejor_leaf)
].copy()

df_tradeoff = df_tradeoff.sort_values(by=['Threshold'])

print("\n=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===")
display(df_mejores)

print(f"\n=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO ({mejor_dia} Días | {mejor_estrategia} | Depth={mejor_depth} | Leaf={mejor_leaf}) ===")
display(df_tradeoff)

Iniciando búsqueda exhaustiva CP-07 (Random Forest)...
--- Evaluando ventana a 30 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 60 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.
--- Evaluando ventana a 90 días ---
  > Línea Base (Sin balanceo): Procesado.
  > Undersampling: Procesado.
  > ADASYN: Procesado.

=== TABLA 1: MÉTRICAS DEL MEJOR MODELO POR ESCENARIO (TEST) ===


,Días,Estrategia,Mejor Max_Depth,Mejor Min_Samples_Leaf,Mejor Threshold,Accuracy,Precision,Recall,F1-Score
0,30,Línea Base (Sin balanceo),None,1,0.3,0.662977,0.501023,0.532226,0.516153
1,30,Undersampling,5,1,0.3,0.365034,0.346116,0.989595,0.512858
2,30,ADASYN,3,1,0.3,0.337757,0.337757,1.000000,0.504960
3,60,Línea Base (Sin balanceo),None,1,0.3,0.666649,0.505690,0.579748,0.540192
4,60,Undersampling,5,5,0.3,0.389740,0.354294,0.980898,0.520564
5,60,ADASYN,3,1,0.3,0.385648,0.353161,0.984780,0.519882
6,90,Línea Base (Sin balanceo),None,1,0.3,0.669219,0.508849,0.593881,0.548087
7,90,Undersampling,5,5,0.3,0.435061,0.370786,0.965057,0.535736
8,90,ADASYN,5,1,0.3,0.404375,0.359751,0.979189,0.526184



=== TABLA 2: PROGRESIÓN DE MÉTRICAS - MEJOR ESCENARIO (30 Días | ADASYN | Depth=3 | Leaf=1) ===


,Días,Estrategia,Max_Depth,Min_Samples_Leaf,Threshold,CV_Accuracy,CV_Precision,CV_Recall,CV_F1
48,30,ADASYN,3,1,0.3,0.307859,0.278995,0.968069,0.432559
49,30,ADASYN,3,1,0.4,0.477865,0.325373,0.827270,0.464481
50,30,ADASYN,3,1,0.5,0.693099,0.498904,0.559329,0.503621
51,30,ADASYN,3,1,0.6,0.757405,0.659059,0.380112,0.459350
